## Introduction

In this tutorial, we will build a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. We will train the model on the text from "warandpeace.txt". This project will help you understand how RNNs can be implemented for text generation tasks and their application in building your own autocomplete model.


## Importing Necessary Libraries

In [1]:
# This is Cell #1

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re
from sklearn.model_selection import train_test_split


## Setting Up the Device

In [2]:
# This is Cell #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Reading and Preprocessing the Data

Now it is time to prepare our training data.


In [3]:
# This is Cell #3

def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

# sequence = read_file("warandpeace.txt")


### Here we will train our model with a simple sequence

We will start by training our model with a simple sequence and repettitive sequence such as `"abcdefghijklmnopqrstuvwxyzabcdef..."`, and we will see if our RNN is capable of learning that pattern or not. This will help you easily verify if your RNN is working correctly or not.

In [4]:
# This is Cell #4

sequence = "abcdefghijklmnopqrstuvwxyz" * 100

## Create Character Mappings

Creating character mappings is essential because RNNs require numerical input to process data. By mapping each unique character to an index and creating a reverse mapping, we convert text data into numerical sequences that the model can understand. This step allows us to encode input text for training and decode the model's output back into readable characters during text generation.



In [15]:
# This is Cell #5

#TODO - DONE: Create a list of unique characters from the text sequence
# space, special char, lowercase a-z
vocab = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', ' ', ".", ",", "!", "?", ";", ":", "(", ")", "[", "]"]

#TODO - DONE: Create two dictionaries for character-index mappings that map each character in vocab to a unique index and vice versa
char_to_idx = {vocab[i] : i for i in range(len(vocab))}
idx_to_char = {i : vocab[i] for i in range(len(vocab))}

#TODO - DONE: Convert the entire text based data into numerical data
data = [char_to_idx[c] for c in sequence]
# print(len(data))
# print(data)


## Defining the CharDataset Class

Now we will create a custom dataset class to generate sequences and targets for training

Creating a custom `CharDataset` class is crucial because it prepares our text data into input sequences and target sequences that the RNN can learn from. By organizing the data this way, we can efficiently feed batches of sequences into the model during training, allowing it to learn the patterns of character sequences in the text.

In [6]:
# This is Cell #6

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []
        
        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target
    

## Setting Hyperparameters

Now we will set our model's hyperparameters for our training process

Setting hyperparameters is important because they define the model's architecture and training behavior. They determine how the RNN processes data, learns patterns, and how quickly it converges during training. Properly chosen hyperparameters can significantly improve model performance and is a key step in training of models

Set the following hyperparameters for your model in the code cell below:
`sequence_length`, `stride`, `embedding_dim`, `hidden_size`, `num_layers`, `learning_rate`, `num_epochs`, `batch_size`, `vocab_size`.

In [16]:
# This is Cell #7

#TODO - DONE: Set your model's hyperparameters

sequence_length = 64  # Length of each input sequence
stride = 10             # Stride for creating sequences
embedding_dim = 64     # Dimension of character embeddings
hidden_size = 256      # Number of features in the hidden state of the RNN
learning_rate = .001  # Learning rate for the optimizer
num_epochs = 2         # Number of epochs to train
batch_size = 64        # Batch size for training
vocab_size = len(vocab)
input_size = len(vocab)
output_size = len(vocab)


After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.

TODO: Explain below
> ADD MORE LATER
> Hyperparameters Explained:

sequence_length: Length of each input sequence. A larger value helps the model understand longer patterns but increases computation.

stride: Step size between starting points of overlapping sequences. Smaller values create more data but with redundancy.

embedding_dim: Size of the embedding vectors for characters. Higher values allow more complex representations but increase computation.

hidden_size: Number of features in the hidden state of the RNN. More features allow the model to capture more complexity.

learning_rate: Determines how much the model updates its weights during training. A value too high may lead to overshooting, too low to slow convergence.

num_epochs: Number of full passes through the dataset. Too few epochs can underfit; too many can overfit.

batch_size: Number of samples processed in one forward/backward pass. Larger batches make training faster but use more memory.

vocab_size: Size of the vocabulary (number of unique characters in the dataset).

input_size and output_size are the sizes of the input and outputs, in this case the length of vocab


## Splitting Data into Training and Testing Sets

By now at this point in class, I'm confident that you know why we do this, so I'm not gonna say a lot here, let's jump right into the todo.

In [17]:
# This is Cell #8

data_tensor = torch.tensor(data, dtype=torch.long)

#TODO - DONE: Convert the data into a pytorch tensor and split the data into 90:10 ratio
train_size = int(0.9 * len(data_tensor))  # 90% of the data
# test_size = len(data_tensor) - train_size  # Remaining 10%
train_data = data_tensor[:train_size]  # First 90% of data
test_data = data_tensor[train_size:]   # Last 10% of data


## Creating Data Loaders

Now we will create data loaders for easy batching during training and testing.

Creating data loaders is essential to batch the data during training and testing. Batching allows the RNN to process multiple sequences in parallel, which speeds up training and makes better use of computational resources. 
We will also use Data loaders to shuffle the batched data, which is important for training models that generalize well.

Make sure to set `drop_last=True`

In [18]:
# This is Cell #9

train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)
print(test_dataset.data)

#TODO: Initialize the training and testing data loader with batching and shuffling equal to True for training (and shuffling = False for testing)
train_loader = DataLoader(train_dataset, batch_size, shuffle = True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size, shuffle = False, drop_last=True)

total_batches = len(train_loader)
# print(len(test_loader))


tensor([14, 17,  4,  ..., 10, 18, 27])


## Defining the RNN Model

Here we will define our character-level RNN model.

In [10]:
# This is Cell #10

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) 
        #TODO: set the fully connected layer
        self.fc = nn.Linear(hidden_size, output_size) # idk if it should be hidden size or input size

    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation from the lecture 
            # We add a bias as well to expand the range of learnable functions
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v] 
        return logits, final_hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)


For a basic high level understanding of what is the CharRNN model that you just defined above, it consists of an embedding layer, an RNN layer, and a fully connected layer. Then embedding layer converts character indices into embeddings. Then RNN processes the embeddings and captures sequential information. Then finally the fully connected layer maps the RNN outputs to the vocabulary size for character prediction.


# Initializing the Model, Loss Function, and Optimizer

Now we will create an instance of the model that we just defined above and set up the loss function and optimizer. Then we will define a loss function, that evaluates the model's prediction against the true targets, and attaches a cost (number) on how good/bad the model is doing. During our training process, it is this cost that we try to minimize by tweaking the weights of the network. 

Then we will set up an optimizer, which will update the model's parameters based on the loss returned by the our loss function. This is how our model will learn over time.


In [19]:
# This is Cell #12

#TODO: Initialize your RNN model
model = CharRNN(input_size, hidden_size, output_size, embedding_dim)

#TODO: Define the loss function (use cross entropy loss)
criterion = nn.CrossEntropyLoss()

#TODO: Initialize your optimizer passing your model parameters and training hyperparameters
optimizer = optim.Adam(model.parameters(), lr = learning_rate)


## Training the Model

Now finally, after all the setup that we have done, we can train our RNN. 

A basic idea high level idea of what we will do here is we will loop over epochs and batches to train the model. 
We will Initialize the hidden state at the beginning of each epoch. For each batch, we will reset the gradients, perform a forward pass, compute the loss, perform backpropagation, and update the model parameters. Then we detach the hidden state to prevent gradients from backpropagating through previous batches. We ill repeat this process for each batch. And finally we will calculate the average loss and accuracy for each epoch.
By performing forward and backward passes, calculating loss, and updating the model parameters, we enable the RNN to improve its predictions with each epoch.

In [20]:
# This is Cell #13

for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/2:   0%|          | 0/4379 [00:00<?, ?it/s]/tmp/ipykernel_3463/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/tmp/ipykernel_3463/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/2: 100%|██████████| 4379/4379 [03:49<00:00, 19.08it/s]


Epoch [1/2], Loss: 1.6247, Accuracy: 52.28%


Epoch 2/2: 100%|██████████| 4379/4379 [03:52<00:00, 18.83it/s]

Epoch [2/2], Loss: 1.4534, Accuracy: 57.12%


## Check your loss

The training loss of your model when trained with a simple sequence like `"abcdefghijklmnopqrstuvwxyz" * 100` should be extremely close to zero. If that's not the case, go back and fix your bugs ;)

If you have acheived a training loss of 0 or extremley close to 0, then congratulations, lets move on to train your model with a bit more complicated sequence. That is our old favorite book, `warandpeace.txt`.

### Read the `warandpeace.txt` file

In [13]:
# This is Cell #14

sequence = read_file('warandpeace.txt')

### Now Follow the instructions

1. Re-run Cell #5 to re-create character mappings for `warandpeace.txt`
2. Re-run Cell #7 to re-initialize hyperparameters
3. Re-run Cell #8 to split and create training and testing data with `warandpeace.txt` as your corpus
4. Re-run Cell #9 to set up data loaders with `warandpeace.txt` data
5. Re-run Cell #12 to re-initialize a new model object (maybe ask yourself why can't you use the previous model that was trained on the simple `"abc..."` corpus)
6. Re-run Cell #13 to train the new model with `warandpeace.txt` data.
   

## Evaluating the Model

After training, we evaluate the model on the test data.

In [21]:
# This is Cell #15

with torch.no_grad():
    #TODO: Write the testing loop for your trained model by refering to the training loop code given to you above
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0
    
    hidden = model.init_hidden(batch_size)  # Initialize hidden state

    # Loop through the test data loader
    for batch_idx, (batch_inputs, batch_targets) in enumerate(test_loader):
        # Move inputs and targets to the device
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)
        
        # Forward pass through the model
        output, hidden = model(batch_inputs, hidden)
        
        # Detach hidden state to prevent unwanted backpropagation
        hidden = hidden.detach()
        
        # Compute the loss
        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))
        total_loss += loss.item()
        
        # Calculate predictions
        predicted_indices = torch.argmax(output, dim=2)  # Get predicted character indices
        correct_predictions += (predicted_indices == batch_targets).sum().item()
        total_predictions += batch_targets.numel()  # Total number of characters

    # Compute average loss and accuracy
    avg_loss = total_loss / len(test_loader)
    accuracy = (correct_predictions / total_predictions) * 100    


    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

/tmp/ipykernel_3463/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/tmp/ipykernel_3463/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)


Test Loss: 1.5103, Test Accuracy: 55.56%


## Generating Text with the Trained Model

In this part of the assignment, your task is to implement the `generate_text` function, which uses a trained RNN model to generate text character-by-character, continuing from a given input. The function will produce an extended sequence by repeatedly predicting and appending the next character to the input.

### What the function is supposed to do?

1. Take an initial input text of length `n` from the user, convert it into indices using a predefined vocabulary (char_to_idx).
2. Use a trained model to predict the next character in the sequence.
3. Append the predicted character to the input, extend the input sequence, and repeat the process until `k` additional characters are generated.
4. Return the generated text, including the original input and the newly predicted characters.


In [22]:
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)
    
    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
        model: The trained RNN model used for character prediction.
        start_text: The initial string of length `n` provided by the user to start the generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: Optional
        A scaling factor for randomness in predictions. Higher values (e.g., >1) make 
            predictions more random, while lower values (e.g., <1) make predictions more deterministic.
            Default is 1.0.
    """
    start_text = start_text.lower()

    #TODO: Implement the rest of the generate_text function
    model.eval()
    input_indices = [char_to_idx[c] for c in start_text]
    input_tensor = torch.tensor(input_indices, dtype=torch.long).unsqueeze(0).to(device)

    hidden = model.init_hidden(batch_size=1)
    generated_indices = input_indices

    for _ in range(k):  # Generate k characters
        # Forward pass through the model
        logits, hidden = model(input_tensor, hidden)
        
        # Get the logits for the last character in the sequence
        # last_logits = logits[0, -1, :]  # Shape: [vocab_size]
        last_logits = logits[0, -1, :].unsqueeze(0)  # Shape: [1, vocab_size]
        
        # Sample the next character index using temperature scaling
        next_index = sample_from_output(last_logits, temperature)
        
        # Append the new character index to the generated sequence
        generated_indices.append(next_index.item())  # Ensure it's an integer
        
        # Prepare the next input (last predicted index)
        input_tensor = torch.tensor([[next_index]], dtype=torch.long).to(device)
    
    # Convert the generated indices back to characters
    generated_text = ''.join([idx_to_char[idx] for idx in generated_indices])
    

    return generated_text

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':
        print("Exiting...")
        break
    
    n = len(start_text) 
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0
    
    completed_text = generate_text(model, start_text, n, k, temperature)
    
    print(f"Generated text: {completed_text}")

Training complete. Now you can generate text.
Generated text: hi! today is a very good day and i would like to do a lot of things today. i was thinking of going to the park, but we could also read on an army for cless and spexpit was kutzov, , that in achien.many wishebergouts important was 
Exiting...


## Report section

In your report, describe your experiments and observations when training the model with two datasets: (1) the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` and (2) the text from `warandpeace.txt`. Include the final loss values for both datasets and discuss how the generated text differed between the two. Explain the impact of changing the `temperature` parameter on the text generation, and provide examples. Reflect on the challenges you faced, your thought process during implementation, and the key insights you gained about RNNs and sequence modeling.

The sequence for `"abcdefghijklmnopqrstuvwxyz" * 100` was much shorter of a corpus than the text from war and peace, so it was more feasible to use multiple epochs to get much higher accuracy. However, I was still able to get good accuracy with 2 epochs for both. It likely would have been more accurate if I used many epochs for war and peace, but it took a while even to run 2 epochs.


#### Training on (1)
When I trained on the sequence `"abcdefghijklmnopqrstuvwxyz" * 100`, I noticed that the generated results were very strange and would produce a sequence of alphabetical letters. For example, I would prompt it with a sentance such as "hello, today is a very good day! today i will also " and ask it to generate 100 characters and had temperature default. The result was `hello, today is a very good day! today i will also jklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabydkfghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabpqw`. Compared to the war and peace corpus, the generated text was way less coherent and didn't produce anything comprehensible. This make sense because the training data for (1) was more akin to a sequence of characters rather than words, so I understand why there were no actual words produced, and instead it just produced letters in alphabetical sequences over and over again.

Note that I had to use a smaller batch size of 16, otherwise similar hyperparameters were used. They are below:

sequence_length = 64  # Length of each input sequence
stride = 10             # Stride for creating sequences
embedding_dim = 64     # Dimension of character embeddings
hidden_size = 256      # Number of features in the hidden state of the RNN
learning_rate = .001  # Learning rate for the optimizer
num_epochs = 2         # Number of epochs to train
batch_size = 16        # Batch size for training
vocab_size = len(vocab)
input_size = len(vocab)
output_size = len(vocab)

The results from epoch 1: Epoch [1/2], Loss: 2.5118, Accuracy: 89.38%
The results from epoch 2: Epoch [2/2], Loss: 0.2889, Accuracy: 98.75%
The resuls from testing: Test Loss: 0.0299, Test Accuracy: 100.00%

When I changed the temperature to be higher, for example at 100, I got things that made a lot less sense. For the same prompt when temperature was 1, the results were alphabetized sequences of letters in alphabetical order. When I did the same prompt but with temperature = 100 instead, this is what it produced `hello, today is a very good day! today i will alsom.nl.hczendlla.xb)agjxrs:epadbwb?rv.o!wziel.fl]skfv)t?dgrp(vrlbx.zuezn[baau?n,kpwc)t]gqlnd :,z;]za[t` which is much more random.

Insights: The model quickly learned the repetitive pattern in the input data. Due to the simplistic nature of the sequence, the loss dropped rapidly, and the model memorized the pattern rather than learning generalized relationships.


#### Training on (2)
When I trained on the corpus from war and peace, I noticed that the generated results consisted of words rather than random letters, and were comprehendible for the most part. Of course, it didn't make complete sense from a logical point, but it was impressive that it did generate words. For example, I prompted it with the same sequence as above: "hello, today is a very good day! today i will also " and asked it to generate 100 characters. When temperature = 1.0, the result was `hello, today is a very good day! today i will also say the warwhy of behind what has long serve were to crossess were to dinger and wolzogement of thed`. Compared to the other dataset, this model wasn't able to run quickly with many epochs and also produced much more coherent results for text generation. Most of the generated words were valid english words and didn't have much random characters or punctuation inserted. However, loss and accuracy were generally worse, even after the second epoch.

The parameters used were:

sequence_length = 64  # Length of each input sequence
stride = 10             # Stride for creating sequences
embedding_dim = 64     # Dimension of character embeddings
hidden_size = 256      # Number of features in the hidden state of the RNN
learning_rate = .001  # Learning rate for the optimizer
num_epochs = 2         # Number of epochs to train
batch_size = 64        # Batch size for training
vocab_size = len(vocab)
input_size = len(vocab)
output_size = len(vocab)

The results from epoch 1: Epoch [1/2], Loss: 1.6255, Accuracy: 52.26%
The results from epoch 2: Epoch [2/2], Loss: 1.4520, Accuracy: 57.10%
The resuls from testing: Test Loss: 1.5101, Test Accuracy: 55.50%

When I changed the temperature to be higher, for example at temperature = 100, I got things that made a lot less sense. When temperature was 1, the results made a bit of sense and still had mostly english words. When I did the same prompt but with temperature = 100 instead, this is what it produced `hello, today is a very good day! a?)dnb?isxxp,[yt]:fkdn!qaffi(t] !g!w](v?.mtrxh, ;.zwh[w vbmpkklxp,zkh[?uyl,;mc[bhw,;igtarcdbcbnb;!yl` which is much more random and doesn't even include valid english. As temperature got higher, the results were more random and nonsensical.

Insights: The model struggled with long-term dependencies and complex sentence structures in warandpeace.txt. Lower temperatures resulted in text closer to the training data but lacked creativity. Higher temperatures produced diverse outputs at the cost of coherence.

#### Temperature
The temperature parameter controls the randomness of predictions during text generation. Low Temperature makes the output deterministic, favoring high-probability characters. Results in repetitive and predictable text. High Temperature increases randomness, allowing for creative but sometimes nonsensical outputs. Useful for exploring the model's creativity but sacrifices coherence.

#### Challenges Faced
Sometimes results took a long time to produce given the resources and time required to run training, especially with larger datasets and more epochs.
Logits Shape Issues: Debugging dimension errors (e.g., IndexError with F.softmax) required careful tensor reshaping.
Hidden State Management: Ensuring the hidden state was reset or detached correctly during generation was critical to avoid backpropagation errors.
Text Quality: Training on warandpeace.txt showed that character-level RNNs struggle with long-term dependencies.

My thoughts in the process were that things didn't always produce the results I thought they would; for example, I was surprised that even with such a large text like war and peace, the generated results were still not great. I did think it's very interesting how RNNs can work so that they can train so many different sizes of data, given how large war and peace was vs how small the sequence of characters was in comparison.

#### Key Insights
Repetitive vs. Complex Data: The RNN easily memorized simple, repetitive patterns but required more training and careful tuning for complex text.
Temperature Effects: Adjusting the temperature parameter effectively balanced diversity and coherence in text generation.
Limitations of RNNs: While RNNs can generate coherent text for short sequences, they struggle with capturing long-term context in real-world datasets.